# Exploratory Data Analysis (EDA) of Student Performance Data

This notebook performs an exploratory data analysis on the `student_data.csv` dataset. We will examine the data's structure, summary statistics, missing values, and relationships between features, particularly focusing on how they might relate to student performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure plots are displayed inline
%matplotlib inline

# Set a consistent style for plots
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
# Define the path to the data file
DATA_PATH = '../data/student_data.csv'

# Create dummy data and directories if they don't exist, similar to data_processing.py
def ensure_dummy_data_exists(path='../data/student_data.csv'):
    data_dir = os.path.dirname(path)
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        print(f"Created directory: {data_dir}")
        
    if not os.path.exists(path):
        print(f"'{path}' not found. Creating a dummy CSV for demonstration.")
        dummy_data = {
            'age': np.random.randint(15, 20, size=100),
            'Medu': np.random.randint(0, 5, size=100),
            'Fedu': np.random.randint(0, 5, size=100),
            'studytime': np.random.randint(1, 5, size=100),
            'failures': np.random.randint(0, 4, size=100),
            'absences': np.random.randint(0, 93, size=100),
            'G1': np.random.randint(0, 20, size=100),
            'G2': np.random.randint(0, 20, size=100),
            'G3': np.random.randint(0, 20, size=100),
            'sex': np.random.choice(['F', 'M'], size=100),
            'address': np.random.choice(['U', 'R'], size=100),
            'internet': np.random.choice(['yes', 'no'], size=100),
            'schoolsup': np.random.choice(['yes', 'no'], size=100),
            'higher': np.random.choice(['yes', 'no'], size=100)
        }
        df_dummy = pd.DataFrame(dummy_data)
        # Introduce some NaNs for missing value analysis demonstration
        for col in ['G1', 'absences', 'internet']:
            idx = df_dummy.sample(frac=0.05, random_state=42).index
            df_dummy.loc[idx, col] = np.nan
        df_dummy.to_csv(path, index=False)
        print(f"Dummy '{path}' created.")
    else:
        print(f"Using existing data file: '{path}'")

ensure_dummy_data_exists(DATA_PATH)

# Load the dataset
try:
    df = pd.read_csv(DATA_PATH)
    print("Student data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file {DATA_PATH} was not found. Please ensure it's in the correct location.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid further errors

## 2. Initial Data Inspection

In [ ]:
if not df.empty:
    print("First 5 rows of the dataset:")
    display(df.head())
else:
    print("DataFrame is empty, skipping head display.")

In [ ]:
if not df.empty:
    print("\nDataset Information (dtypes, non-null counts):")
    df.info()
else:
    print("DataFrame is empty, skipping info display.")

In [ ]:
if not df.empty:
    print("\nSummary Statistics:")
    display(df.describe(include='all'))
else:
    print("DataFrame is empty, skipping describe display.")

In [ ]:
if not df.empty:
    print("\nMissing Value Counts:")
    missing_counts = df.isnull().sum()
    print(missing_counts[missing_counts > 0])
    if missing_counts.sum() == 0:
        print("No missing values found.")
else:
    print("DataFrame is empty, skipping missing value check.")

## 3. Define Target Variable ('passed')

In [ ]:
if not df.empty:
    # Define 'passed' column based on 'G3' (final grade) if it doesn't exist
    # This logic mirrors what's in data_processing.py for consistency
    if 'passed' not in df.columns:
        if 'G3' in df.columns:
            df['passed'] = (df['G3'] >= 10).astype(int)
            print("Created 'passed' column (1 if G3 >= 10, else 0).")
        else:
            print("Warning: 'G3' column not found, cannot create 'passed' column. Some plots might not work.")
            # Create a dummy 'passed' column if G3 is also missing, for placeholder plots
            if 'passed' not in df.columns : df['passed'] = np.random.choice([0,1], size=len(df))
    else:
        print("Using existing 'passed' column.")
    
    if 'passed' in df.columns:
        print("\nValue counts for 'passed':")
        print(df['passed'].value_counts(normalize=True))
else:
    print("DataFrame is empty, skipping target variable creation.")

## 4. Visualizations

Ensure `reports/figures` directory exists for saving plots.

In [ ]:
FIGURES_DIR = '../reports/figures/'
if not os.path.exists(FIGURES_DIR):
    os.makedirs(FIGURES_DIR)
    print(f"Created directory: {FIGURES_DIR}")

### 4.1 Histograms of Numeric Features

In [ ]:
if not df.empty:
    # Identify potential numeric columns (as per issue, e.g., age, absences, G1, G2)
    # For robustness, select columns of numeric type from the dataframe
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    # Remove 'passed' if it was added and is numeric, G3 might also be removed if not primary grade
    if 'passed' in numeric_cols: numeric_cols.remove('passed')
    # Specific columns mentioned in issue:
    hist_cols = [col for col in ['age', 'absences', 'G1', 'G2', 'G3'] if col in numeric_cols and col in df.columns]
    
    if hist_cols:
        print(f"Plotting histograms for: {hist_cols}")
        df[hist_cols].hist(bins=15, figsize=(15, 10), layout=(-1, 3))
        plt.suptitle('Histograms of Key Numeric Features', y=1.02, fontsize=16)
        # Save the figure
        # fig_path = os.path.join(FIGURES_DIR, 'numeric_features_histograms.png')
        # plt.savefig(fig_path)
        # print(f"Saved histograms to {fig_path}")
        plt.show()
    else:
        print("No suitable numeric columns found for histograms (expected 'age', 'absences', 'G1', 'G2', 'G3').")
else:
    print("DataFrame is empty, skipping histograms.")

### 4.2 Bar Charts for Categorical Features vs. Target ('passed')

In [ ]:
if not df.empty and 'passed' in df.columns:
    # Identify potential categorical columns (e.g., gender, address, study_time categories)
    # For robustness, select columns of object or category type
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    # Example categorical columns from typical student datasets, filter by what's available
    bar_chart_cols = [col for col in ['sex', 'address', 'studytime', 'schoolsup', 'internet', 'higher'] if col in categorical_cols and col in df.columns]

    if bar_chart_cols:
        print(f"Plotting bar charts for: {bar_chart_cols} vs 'passed'")
        num_plots = len(bar_chart_cols)
        num_cols = 3 # Number of columns for subplots
        num_rows = (num_plots + num_cols - 1) // num_cols # Calculate rows needed
        
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows * 5))
        axes = axes.flatten() # Flatten in case of single row/column

        for i, col in enumerate(bar_chart_cols):
            sns.countplot(x=col, hue='passed', data=df, ax=axes[i], palette='viridis')
            axes[i].set_title(f'{col} vs. Passed')
            axes[i].tick_params(axis='x', rotation=45)
        
        # Hide any unused subplots
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
            
        plt.tight_layout()
        # fig_path = os.path.join(FIGURES_DIR, 'categorical_features_vs_target.png')
        # plt.savefig(fig_path)
        # print(f"Saved bar charts to {fig_path}")
        plt.show()
    else:
        print("No suitable categorical columns found for bar charts or 'passed' column is missing.")
elif df.empty:
    print("DataFrame is empty, skipping bar charts.")
else:
    print("'passed' column not in DataFrame, skipping categorical bar charts vs target.")

### 4.3 Correlation Heatmap for Numeric Features

In [ ]:
if not df.empty:
    numeric_df = df.select_dtypes(include=np.number)
    if not numeric_df.empty:
        plt.figure(figsize=(12, 10))
        correlation_matrix = numeric_df.corr()
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
        plt.title('Correlation Heatmap of Numeric Features (including target if numeric)', fontsize=16)
        fig_path = os.path.join(FIGURES_DIR, 'correlation_heatmap.png')
        plt.savefig(fig_path)
        print(f"Saved correlation heatmap to {fig_path}")
        plt.show()
    else:
        print("No numeric features found for correlation heatmap.")
else:
    print("DataFrame is empty, skipping correlation heatmap.")

### 4.4 Boxplots for G1 and G2 Grouped by Passed vs. Failed

In [ ]:
if not df.empty and 'passed' in df.columns and 'G1' in df.columns and 'G2' in df.columns:
    grade_cols_for_boxplot = ['G1', 'G2']
    fig, axes = plt.subplots(1, len(grade_cols_for_boxplot), figsize=(12, 6), sharey=True)
    
    if len(grade_cols_for_boxplot) == 1: # Make axes iterable if only one plot
        axes = [axes]

    for i, grade_col in enumerate(grade_cols_for_boxplot):
        sns.boxplot(x='passed', y=grade_col, data=df, ax=axes[i], palette='pastel')
        axes[i].set_title(f'{grade_col} Distribution by Performance')
        axes[i].set_xticklabels(['Failed (0)', 'Passed (1)'])
        
    plt.suptitle('Boxplots of G1 and G2 by Passing Status', fontsize=16, y=1.03)
    # fig_path = os.path.join(FIGURES_DIR, 'G1_G2_boxplots_by_passed.png')
    # plt.savefig(fig_path)
    # print(f"Saved boxplots to {fig_path}")
    plt.tight_layout()
    plt.show()
elif df.empty:
    print("DataFrame is empty, skipping boxplots.")
else:
    print("Required columns ('passed', 'G1', 'G2') not available for boxplots.")

## 5. Document Observations

Based on the visualizations (especially the correlation heatmap and bar charts vs. 'passed'):

1.  **Correlations with Grades (G1, G2, G3):** 
    *   `G1` and `G2` are typically highly correlated with `G3` and thus with 'passed'. Students who do well in earlier periods tend to do well overall.
    *   `studytime` often shows a positive correlation with grades, though it might not be very strong.
    *   `failures` (number of past class failures) usually has a strong negative correlation with grades and passing status.

2.  **Categorical Feature Insights:**
    *   `higher` (wants to take higher education): Students aspiring for higher education often perform better.
    *   `schoolsup` / `famsup` (extra educational support): The impact can vary; sometimes students receiving support are those already struggling.
    *   `internet` access might show a positive association with performance.
    *   Parental education (`Medu`, `Fedu`) can also be positively correlated with student success.

3.  **Other Numeric Features:**
    *   `absences`: Higher absences are generally linked to poorer performance.
    *   `age`: In some datasets, older students in a cohort (possibly due to repeating years) might show lower average performance.

*(These are general expectations. The actual observations will depend on the specific `student_data.csv` used. The dummy data will produce random-looking plots and correlations.)*